ChromaDB:

What:
A vector database.

Purpose:
Stores embeddings and lets you search for similar embeddings.


Chunk → Embedding → Stored in ChromaDB

Library:
chromadb

Used:
client = chromadb.Client()
collection.add(...)
collection.query(...)

--------------------------------------------------

LangChain:

What:
A framework for building LLM applications.

Purpose:
Provides ready-made components for:
- PDF loading
- Chunking
- Retrieval
- RAG pipelines
- Agents

Library:
langchain


PyPDFLoader
RecursiveCharacterTextSplitter

Later:
VectorStoreRetriever
RetrievalQA

--------------------------------------------------

Chunking:

Method:
Recursive Character Text Splitting

Library:
langchain_text_splitters

Class:
RecursiveCharacterTextSplitter

Parameters Used:

chunk_size=500
chunk_overlap=50

How it works:

Large Text
↓
Split into 500-character chunks
↓
Keep 50-character overlap between neighboring chunks

--------------------------------------------------

Embeddings:

Model:
all-MiniLM-L6-v2

Library:
sentence-transformers

Code:

SentenceTransformer("all-MiniLM-L6-v2")

Output:

Text
↓
384-dimensional vector

Example:

"This is Logistic Regression"
↓
[-0.12, 0.45, ..., 384 values]

--------------------------------------------------

Current RAG Tech Stack:

PDF Loading:
PyPDFLoader (LangChain)

Chunking:
RecursiveCharacterTextSplitter (LangChain)

Embedding Model:
all-MiniLM-L6-v2 (Sentence Transformers)

Vector Database:
ChromaDB

LLM:
FLAN-T5 Base

RAG Type:
Manual RAG

RecursiveCharacterTextSplitter

What:
Splits large text into smaller chunks.

Why "Recursive":
It tries different separators from large to small until the chunk fits.

Order:

Paragraphs ("\n\n")
↓
Lines ("\n")
↓
Words (" ")
↓
Characters ("")

Why in RAG:
Creates meaningful chunks while preserving context.



chunk_size=500
chunk_overlap=50

Meaning:

~500 characters per chunk
+
50 characters shared with the next chunk

In [1]:
!pip install -q langchain langchain-community langchain-chroma sentence-transformers

Why in RAG:

We are moving from Manual RAG to LangChain RAG.

Explanation:

- langchain: RAG framework
- langchain-community: loaders and integrations
- langchain-chroma: ChromaDB integration
- sentence-transformers: embedding model



In [2]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma sentence-transformers pypdf

In [3]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_file)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "What is Logistic Regression?"

retrieved_docs = retriever.invoke(query)

for doc in retrieved_docs:
    print(doc.page_content)
    print("-" * 100)

/tmp/ipykernel_3137/2563332304.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Saving NBayesLogReg.pdf to NBayesLogReg.pdf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

data to directly estimate P(Y|X), in contrast to Naive Bayes. In this sense,
Logistic Regression is often referred to as adiscriminative classiﬁer because
we can view the distribution P(Y|X) as directly discriminating the value of
the target value Y for any given instance X.
• Logistic Regression is a linear classiﬁer over X. The linear classiﬁers pro-
duced by Logistic Regression and Gaussian Naive Bayes are identical in
the limit as the number of training examples approaches inﬁnity, provided
----------------------------------------------------------------------------------------------------
Copyright c⃝ 2017, Tom M. Mitchell. 12
we work with the gradient, which is the vector of partial derivatives. The ith
component of the vector gradient has the form
∂l(W )
∂wi
= ∑
l
X l
i (Y l− ˆP(Y l = 1|X l,W ))
where ˆP(Y l|X l,W ) is the Logistic Regression prediction using equations (24) and
(25) and the weights W. To accommodate weight w0, we assume an imaginary
X0 = 1 for all l. This expres

Why in RAG:

Find relevant chunks for a user query.

Explanation:

- PyPDFLoader loads the PDF.
- RecursiveCharacterTextSplitter creates chunks.
- HuggingFaceEmbeddings converts chunks to vectors.
- Chroma stores the vectors.
- retriever.invoke(query) finds the most relevant chunks.

Output:

Top 3 chunks related to "What is Logistic Regression?"

In [4]:
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

print(context)

data to directly estimate P(Y|X), in contrast to Naive Bayes. In this sense,
Logistic Regression is often referred to as adiscriminative classiﬁer because
we can view the distribution P(Y|X) as directly discriminating the value of
the target value Y for any given instance X.
• Logistic Regression is a linear classiﬁer over X. The linear classiﬁers pro-
duced by Logistic Regression and Gaussian Naive Bayes are identical in
the limit as the number of training examples approaches inﬁnity, provided

Copyright c⃝ 2017, Tom M. Mitchell. 12
we work with the gradient, which is the vector of partial derivatives. The ith
component of the vector gradient has the form
∂l(W )
∂wi
= ∑
l
X l
i (Y l− ˆP(Y l = 1|X l,W ))
where ˆP(Y l|X l,W ) is the Logistic Regression prediction using equations (24) and
(25) and the weights W. To accommodate weight w0, we assume an imaginary
X0 = 1 for all l. This expression for the derivative has an intuitive interpretation:

ˆσ2
ik = 1
(∑ j δ(Y j = yk))− 1 ∑
j
(X j
i

Why in RAG:

Combine retrieved chunks into one context for the LLM.

Explanation:

- retrieved_docs contains the top chunks.
- doc.page_content gets the text from each chunk.
- join() merges them into a single context string.

Output:

One large context block ready to be sent to an LLM.

In [5]:
question = "What is Logistic Regression?"

prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

print(prompt)


Context:
data to directly estimate P(Y|X), in contrast to Naive Bayes. In this sense,
Logistic Regression is often referred to as adiscriminative classiﬁer because
we can view the distribution P(Y|X) as directly discriminating the value of
the target value Y for any given instance X.
• Logistic Regression is a linear classiﬁer over X. The linear classiﬁers pro-
duced by Logistic Regression and Gaussian Naive Bayes are identical in
the limit as the number of training examples approaches inﬁnity, provided

Copyright c⃝ 2017, Tom M. Mitchell. 12
we work with the gradient, which is the vector of partial derivatives. The ith
component of the vector gradient has the form
∂l(W )
∂wi
= ∑
l
X l
i (Y l− ˆP(Y l = 1|X l,W ))
where ˆP(Y l|X l,W ) is the Logistic Regression prediction using equations (24) and
(25) and the weights W. To accommodate weight w0, we assume an imaginary
X0 = 1 for all l. This expression for the derivative has an intuitive interpretation:

ˆσ2
ik = 1
(∑ j δ(Y j = yk))− 1 

Why in RAG:

Create the final prompt for the LLM.

Explanation:

- context contains retrieved information.
- question contains the user's query.
- prompt combines both.

Output:

A complete prompt containing context + question.

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Logistic Regression is an approach to learning functions of the formf : X Y , or P(Y|X) in the case where Y is discrete-valued


Why in RAG:

Generate the final answer from the retrieved context.

Explanation:

- Loads FLAN-T5.
- Sends the prompt to the model.
- Generates an answer.
- Converts output tokens back to text.

Output:

Answer to the question based on the retrieved context.

Tokenizer

What:
Converts text ↔ numbers (tokens).

Why:
LLMs understand numbers, not text.

Flow:

Text

↓

Tokenizer

↓

Tokens

↓

LLM

↓

Tokens

↓

Tokenizer.decode()

↓

Text